In [ ]:
import os
import requests
import pandas as pd
import datetime
import pyarrow

# Передаем сегодняшнюю дату 
load_date = datetime.date.today()
load_date_start = "2025-11-15"
load_date_end = datetime.date.today() - datetime.timedelta(days=1)

TOKEN = "y0__xDe_Im4BBiirjkg2Zzs_xP3srDDcLIeER6ns2I71_wDiFmYXg"
COUNTER_ID = 103580753

headers = {
    "Authorization": f"OAuth {TOKEN}"
}

metrics = [
    "ym:s:visits",
    "ym:s:users",
    "ym:s:newUsers",
    "ym:s:pageviews"
]

# Измерения:
dimensions = [
    "ym:s:date",
    "ym:s:regionCountry",
    "ym:s:regionCity",
    "ym:s:browserLanguage", 
    "ym:s:browser",  # Наименование браузера
    "ym:s:deviceCategory",  # Тип устройства (desktop/mobile/tablet)
    "ym:s:operatingSystem",  # Операционная система
    "ym:s:hour",
]

params = {
    "date1": load_date_start,
    "date2": load_date_end,
    "ids": COUNTER_ID,
    "metrics": ",".join(metrics),
    "dimensions": ",".join(dimensions)
}

url = "https://api-metrika.yandex.net/stat/v1/data"

response = requests.get(url, headers=headers, params=params)
data = response.json()

dimension_names = data["query"]["dimensions"]
metric_names = data["query"]["metrics"]

# Строим список строк
rows = []
for row in data["data"]:
    dim_values = [d["name"] for d in row["dimensions"]]
    metrics = row["metrics"]
    rows.append(dim_values + metrics)

# Названия колонок
columns = dimension_names + metric_names

# Финальный DataFrame
df = pd.DataFrame(rows, columns=columns)

df["update_at"] = load_date

print(df)

# Генерируем имена файлов
filename_csv = f"avpalatov_api_yandex_response_{load_date}.csv"
filename_parquet = f"avpalatov_api_yandex_response_{load_date}.parquet"


# Сохранить в CSV и parquet
if len(df) > 0:
    # Сохранение в файл CSV без индекса
    df.to_csv(filename_csv, sep=';', header=True, index=False) #csv
    df.to_parquet(filename_parquet, index=False) #parquet
